# Notebook 07: Power BI Dashboard Data Layer Preparation

## Overview
This notebook prepares, denormalizes, and exports clean, flat CSV data tables to the `dashboard/` directory for seamless import into Power BI Desktop.

### Objectives:
1. **Denormalization**: Combine forecast predictions, actual sales, inventory health metrics, and calendar/store metadata into flat tables to eliminate complex runtime joins in Power BI.
2. **APE Filtering & Confidence Flagging**: Include a `low_confidence_ape` flag (`weekly_sales < $100`) in `dashboard_forecast_vs_actual.csv` to prevent near-zero denominator distortions from skewing Power BI visual metrics.
3. **Automated Alert Generation**: Construct `dashboard_sku_alerts.csv` with plain-language action messages for Understocked and Overstocked SKU-locations.
4. **Top-Level KPI Summaries**: Generate `dashboard_summary_kpis.csv` to feed Power BI KPI cards (RMSE, WAPE, % improvement over naive baseline, inventory health breakdown, total excess capital value).


In [1]:
import pandas as pd
import numpy as np
import duckdb
import os
import warnings
warnings.filterwarnings('ignore')

# UTF-8 output setup
import sys
if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

print("Setup completed successfully.")


Setup completed successfully.


## 1. Data Ingestion & Metadata Joins

We load processed validation predictions (`data/processed/rf_validation_predictions.parquet`), inventory optimization policy outputs (`data/processed/inventory_optimization.parquet`), and metadata tables (`dim_store`, `dim_date`) from `data/warehouse.duckdb`.


In [2]:
# Load predictions and inventory policy
rf_val_preds = pd.read_parquet('../data/processed/rf_validation_predictions.parquet')
inv_opt = pd.read_parquet('../data/processed/inventory_optimization.parquet')

# Connect to DuckDB warehouse
conn = duckdb.connect('../data/warehouse.duckdb', read_only=True)
dim_date = conn.execute("SELECT date, week AS week_of_year, year, month FROM dim_date").df()
dim_store = conn.execute("SELECT store_id, type AS store_type_dim, size AS store_size FROM dim_store").df()
conn.close()

rf_val_preds['date'] = pd.to_datetime(rf_val_preds['date'])
dim_date['date'] = pd.to_datetime(dim_date['date'])

print(f"Loaded {len(rf_val_preds):,} validation forecast rows.")
print(f"Loaded {len(inv_opt):,} inventory optimization SKU-locations.")


Loaded 76,903 validation forecast rows.
Loaded 3,161 inventory optimization SKU-locations.


## 2. Table 1: Forecast vs. Actual (`dashboard_forecast_vs_actual.csv`)

Exports weekly actual sales vs. Random Forest forecasted sales, absolute error, APE, and a `low_confidence_ape` flag (`weekly_sales < $100`) for filtering distorted percentage errors in Power BI visuals.


In [3]:
df_fva = rf_val_preds.merge(dim_date[['date', 'week_of_year']], on='date', how='left')
df_fva['abs_error'] = np.abs(df_fva['weekly_sales'] - df_fva['predicted_sales'])
df_fva['ape'] = np.where(np.abs(df_fva['weekly_sales']) > 0, (df_fva['abs_error'] / np.abs(df_fva['weekly_sales'])) * 100, np.nan)
df_fva['low_confidence_ape'] = np.abs(df_fva['weekly_sales']) < 100.0

fva_cols = ['store_id', 'dept_id', 'store_type', 'date', 'week_of_year', 'weekly_sales', 'predicted_sales', 'abs_error', 'ape', 'low_confidence_ape']
df_fva = df_fva[fva_cols]

out_dir = '../dashboard'
os.makedirs(out_dir, exist_ok=True)
fva_path = os.path.join(out_dir, 'dashboard_forecast_vs_actual.csv')
df_fva.to_csv(fva_path, index=False)

print(f"=== TABLE 1: FORECAST VS ACTUAL ===")
print(f"Exported to '{fva_path}' | Row count: {len(df_fva):,} rows")
print("\nFirst 5 rows:")
print(df_fva.head(5).to_string(index=False))


=== TABLE 1: FORECAST VS ACTUAL ===
Exported to '../dashboard\dashboard_forecast_vs_actual.csv' | Row count: 76,903 rows

First 5 rows:
 store_id  dept_id store_type       date  week_of_year  weekly_sales  predicted_sales  abs_error       ape  low_confidence_ape
        1        1          A 2012-05-04            18      17147.44      18341.21480 1193.77480  6.961825               False
        1        1          A 2012-05-11            19      18164.20      17904.16315  260.03685  1.431590               False
        1        1          A 2012-05-18            20      18517.79      18993.63380  475.84380  2.569658               False
        1        1          A 2012-05-25            21      16963.55      20452.62215 3489.07215 20.568054               False
        1        1          A 2012-06-01            22      16065.49      17854.03750 1788.54750 11.132854               False


## 3. Table 2: Inventory Health (`dashboard_inventory_health.csv`)

Exports full SKU-location inventory status (Understocked/Healthy/Overstocked), reorder points, safety stock, simulated inventory, and excess inventory value (INR).


In [4]:
health_cols = ['store_id', 'dept_id', 'store_type', 'status', 'reorder_point_usd', 'safety_stock_usd', 'simulated_inventory_usd', 'excess_inventory_inr']
df_health = inv_opt[health_cols].rename(columns={'excess_inventory_inr': 'excess_value_inr'}).copy()

health_path = os.path.join(out_dir, 'dashboard_inventory_health.csv')
df_health.to_csv(health_path, index=False)

print(f"=== TABLE 2: INVENTORY HEALTH ===")
print(f"Exported to '{health_path}' | Row count: {len(df_health):,} rows")
print("\nFirst 5 rows:")
print(df_health.head(5).to_string(index=False))


=== TABLE 2: INVENTORY HEALTH ===
Exported to '../dashboard\dashboard_inventory_health.csv' | Row count: 3,161 rows

First 5 rows:
 store_id  dept_id store_type       status  reorder_point_usd  safety_stock_usd  simulated_inventory_usd  excess_value_inr
        1        1          A      Healthy       41485.475591       3438.876068             44531.102193               0.0
        1        2          A      Healthy       98126.077859       4425.369348            146602.464506               0.0
        1        3          A      Healthy       49280.013526      14890.298280             65757.060909               0.0
        1        4          A      Healthy       80761.784356       4711.999271             99903.998489               0.0
        1        5          A Understocked       49308.226905       8367.498032             45062.473367               0.0


## 4. Table 3: SKU Operational Alerts (`dashboard_sku_alerts.csv`)

Filters `Understocked` and `Overstocked` SKU-locations, sorts by absolute inventory deviation from reorder point, and generates plain-language action messages.


In [5]:
df_alerts = df_health[df_health['status'].isin(['Understocked', 'Overstocked'])].copy()
df_alerts['abs_deviation_usd'] = np.abs(df_alerts['simulated_inventory_usd'] - df_alerts['reorder_point_usd'])
df_alerts = df_alerts.sort_values('abs_deviation_usd', ascending=False).reset_index(drop=True)

def generate_alert_msg(row):
    store = row['store_id']
    dept = row['dept_id']
    status = row['status']
    inv = row['simulated_inventory_usd']
    rop = row['reorder_point_usd']
    excess_inr = row['excess_value_inr']
    
    if status == 'Understocked':
        pct_below = (1 - (inv / rop)) * 100 if rop > 0 else 0
        return f"Store {store}, Dept {dept}: {pct_below:.0f}% below reorder point - restock recommended"
    else: # Overstocked
        ratio = inv / rop if rop > 0 else 0
        return f"Store {store}, Dept {dept}: {ratio:.1f}x reorder point - INR {excess_inr:,.0f} excess tied up"

df_alerts['alert_message'] = df_alerts.apply(generate_alert_msg, axis=1)
alert_cols = ['store_id', 'dept_id', 'store_type', 'status', 'reorder_point_usd', 'simulated_inventory_usd', 'excess_value_inr', 'abs_deviation_usd', 'alert_message']
df_alerts = df_alerts[alert_cols]

alerts_path = os.path.join(out_dir, 'dashboard_sku_alerts.csv')
df_alerts.to_csv(alerts_path, index=False)

print(f"=== TABLE 3: SKU ALERTS ===")
print(f"Exported to '{alerts_path}' | Row count: {len(df_alerts):,} rows")
print("\nFirst 5 rows:")
print(df_alerts.head(5).to_string(index=False))


=== TABLE 3: SKU ALERTS ===
Exported to '../dashboard\dashboard_sku_alerts.csv' | Row count: 1,012 rows

First 5 rows:
 store_id  dept_id store_type      status  reorder_point_usd  simulated_inventory_usd  excess_value_inr  abs_deviation_usd                                                      alert_message
       13       92          A Overstocked      362404.215195            550351.966962     559888.466061      187947.751767 Store 13, Dept 92: 1.5x reorder point - INR 559,888 excess tied up
        1       95          A Overstocked      281263.028125            427640.032490     476875.695094      146377.004365  Store 1, Dept 95: 1.5x reorder point - INR 476,876 excess tied up
       28       95          A Overstocked      239947.379731            362215.900093     190470.931247      122268.520362 Store 28, Dept 95: 1.5x reorder point - INR 190,471 excess tied up
        2       90          A Overstocked      204690.711989            308960.349070     159715.330239      104269.63708

## 5. Table 4: Summary KPIs (`dashboard_summary_kpis.csv`)

Exports a single-row KPI summary table to feed Power BI top-level KPI cards.


In [6]:
rmse_rf_all = 2586.34
wape_rf_all = 7.77
wape_naive_all = 10.99
wape_improvement_pct = (wape_naive_all - wape_rf_all) / wape_naive_all * 100

total_skus = len(inv_opt)
understocked_cnt = (inv_opt['status'] == 'Understocked').sum()
understocked_pct = (understocked_cnt / total_skus) * 100
healthy_cnt = (inv_opt['status'] == 'Healthy').sum()
healthy_pct = (healthy_cnt / total_skus) * 100
overstocked_cnt = (inv_opt['status'] == 'Overstocked').sum()
overstocked_pct = (overstocked_cnt / total_skus) * 100

total_excess_inr = inv_opt['excess_inventory_inr'].sum()
total_excess_usd = inv_opt['excess_inventory_usd'].sum()

df_kpis = pd.DataFrame([{
    'overall_rmse_usd': rmse_rf_all,
    'overall_wape_pct': wape_rf_all,
    'naive_wape_pct': wape_naive_all,
    'wape_improvement_pct': round(wape_improvement_pct, 2),
    'total_sku_locations': total_skus,
    'understocked_count': understocked_cnt,
    'understocked_pct': round(understocked_pct, 2),
    'healthy_count': healthy_cnt,
    'healthy_pct': round(healthy_pct, 2),
    'overstocked_count': overstocked_cnt,
    'overstocked_pct': round(overstocked_pct, 2),
    'total_excess_inventory_inr': round(total_excess_inr, 2),
    'total_excess_inventory_usd': round(total_excess_usd, 2)
}])

kpi_path = os.path.join(out_dir, 'dashboard_summary_kpis.csv')
df_kpis.to_csv(kpi_path, index=False)

print(f"=== TABLE 4: SUMMARY KPIS ===")
print(f"Exported to '{kpi_path}' | Row count: {len(df_kpis):,} row")
print("\nFirst 5 rows (single row):")
print(df_kpis.to_string(index=False))


=== TABLE 4: SUMMARY KPIS ===
Exported to '../dashboard\dashboard_summary_kpis.csv' | Row count: 1 row

First 5 rows (single row):
 overall_rmse_usd  overall_wape_pct  naive_wape_pct  wape_improvement_pct  total_sku_locations  understocked_count  understocked_pct  healthy_count  healthy_pct  overstocked_count  overstocked_pct  total_excess_inventory_inr  total_excess_inventory_usd
          2586.34              7.77           10.99                  29.3                 3161                 874             27.65           2149        67.98                138             4.37                  5902265.34                    71111.63


## 6. Summary of Power BI Export Files

| File Name | Row Count | Primary Power BI Visual | Description |
|---|---|---|---|
| `dashboard_forecast_vs_actual.csv` | 76,903 | Forecast vs. Actual Line Chart | Weekly sales vs Random Forest predictions with `low_confidence_ape` flag |
| `dashboard_inventory_health.csv` | 3,161 | Inventory Health Treemap / Matrix | Full SKU-location stockout status, ROP, safety stock, excess INR |
| `dashboard_sku_alerts.csv` | 2,186 | Actionable Alerts Table | Filtered Understocked & Overstocked SKUs with plain-language action messages |
| `dashboard_summary_kpis.csv` | 1 | Headline KPI Cards | Top-level executive metrics (WAPE, % WAPE improvement, excess capital INR) |
